# E-LSTM SOH forecasting - step by step
Reproduces Arbaoui et al. (Energy and AI 17, 2024) following the authors' public repo.

**Before starting:** Runtime > Change runtime type > **T4 GPU**. Then run the cells in order.

## Step 1 - Check GPU and libraries

In [ ]:
!nvidia-smi -L
!pip -q install h5py scipy scikit-learn matplotlib
import tensorflow as tf
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

## Step 2 - Check the 3 data files
You need all three `*batchdata*.mat` files somewhere under `/content`. If the list is empty, re-download them (Colab resets `/content` when the runtime restarts).

In [ ]:
import glob, os
files = sorted(glob.glob('/content/**/*batchdata*.mat', recursive=True))
for f in files: print(os.path.basename(f), round(os.path.getsize(f)/1e9, 2), 'GB')
assert len(files) == 3, 'Expected 3 files: 2017-05-12, 2017-06-30, 2018-04-12'

## Step 3 - Imports

In [ ]:
import os, glob, pickle
import numpy as np
import h5py
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from scipy.signal import savgol_filter
from sklearn.model_selection import KFold, GroupKFold

tf.random.set_seed(42); np.random.seed(42)

## Step 4 - Settings
`IN_STEP, OUT_STEP`: (10,10) = Table 3, (25,25) = Table 4, (25,50) = Table 5.
`AE_EPOCHS`: auto-encoder epochs (repo uses up to 300 with early stopping).

In [ ]:
# ------------------------------- CONFIG -----------------------------------
IN_STEP, OUT_STEP = 10, 10        # (10,10) Table 3 | (25,25) Table 4 | (25,50) Table 5
LEN = 500                         # fixed feature length (zero-pad / truncate), as in repo
FSIZE = 64                        # encoder code size per cycle (640 = 64*10)
AE_EPOCHS = int(os.environ.get('AE_EPOCHS', 30))     # repo: 300 with early stopping
AE_STRIDE = 2                     # use every 2nd train window for the auto-encoder (speed)
LSTM_EPOCHS = int(os.environ.get('LSTM_EPOCHS', 1))  # repo calls model.fit() with default = 1 epoch
SYNTH = os.environ.get('SYNTH') == '1'               # tiny fake data, only for a quick self-test
CACHE = '/content/authors_cells.pkl'
MAT_FILES = sorted(glob.glob('/content/**/*batchdata*.mat', recursive=True))
PAPER = {(10, 10): [0.86, 0.17, 1.30, 0.91],   # MAE x1e-2, MSE x1e-3, RMSE x1e-2, MAPE %
         (25, 25): [0.83, 0.16, 1.26, 0.89],
         (25, 50): [0.89, 0.19, 1.37, 0.95]}
print('window:', (IN_STEP, OUT_STEP), '| files:', MAT_FILES)

## Step 5 - Loading functions

In [ ]:
# ----------------------------- LOADING ------------------------------------
def pad(a):
    out = np.zeros(LEN, np.float32); m = min(LEN, len(a)); out[:m] = a[:m]
    return out, m

def load_mat(path):
    cells = []
    with h5py.File(path, 'r') as f:
        batch = f['batch']; n_cells = batch['summary'].shape[0]
        for k in range(n_cells):
            try:
                qd = np.hstack(f[batch['summary'][k, 0]]['QDischarge'][0, :].tolist()).astype(np.float64)
                cyc = f[batch['cycles'][k, 0]]
                ncyc = min(len(qd), cyc['I'].shape[0])
            except Exception:
                continue
            X, Ls, soh, lo, hi = [], [], [], [], []
            for j in range(1, ncyc):                      # repo starts at cycle 1
                try:
                    I = f[cyc['I'][j, 0]][()].flatten(); V = f[cyc['V'][j, 0]][()].flatten()
                    T = f[cyc['T'][j, 0]][()].flatten()
                    sd = np.where(I < -0.05)[0][0]; ed = np.where(I < -3.9)[0][-1]
                    ch = np.where(I > 0)[0]; sc, ec = ch[0], ch[-1]
                    feats = [I[sc:ec], I[sd:ed], V[sc:ec], V[sd:ed], T[sc:ec], T[sd:ed]]  # Ic,Id,Vc,Vd,Tc,Td
                    if min(len(a) for a in feats) < 2: continue
                    P = [pad(a) for a in feats]
                    X.append(np.stack([p[0] for p in P])); Ls.append([p[1] for p in P])
                    lo.append([a.min() for a in feats]); hi.append([a.max() for a in feats])
                    soh.append(qd[j] / qd[1])             # SOH = QD(j) / QD(1)   (repo)
                except Exception:
                    continue
            if len(soh) > 0:
                cells.append(dict(X=np.stack(X), n=np.array(Ls, np.int16), soh=np.array(soh, np.float64),
                                  mn=np.min(lo, axis=0), mx=np.max(hi, axis=0)))
            print(f'{os.path.basename(path)[:10]} cell {k+1}/{n_cells}: {len(soh)} cycles   ', end='\r')
    return cells

def make_synth(n_cells=12):
    rng = np.random.default_rng(0); cells = []
    for _ in range(n_cells):
        n = int(rng.integers(150, 260)); t = np.linspace(0, 1, n)
        soh = 1 - 0.18 * t ** 2 + 0.003 * rng.standard_normal(n)
        X = rng.random((n, 6, LEN)).astype(np.float32) * (1 - 0.1 * t)[:, None, None]
        cells.append(dict(X=X, n=np.full((n, 6), 400, np.int16), soh=soh,
                          mn=np.zeros(6), mx=np.ones(6)))
    return cells

## Step 6 - Load all cells (first run takes ~20-30 min, then it is cached)

In [ ]:
if SYNTH:
    cells = make_synth()
elif os.path.exists(CACHE):
    cells = pickle.load(open(CACHE, 'rb'))
else:
    cells = []
    for p in MAT_FILES: cells += load_mat(p)
    pickle.dump(cells, open(CACHE, 'wb'))
print(f'\ncells loaded: {len(cells)}')
assert len(cells) > 5, 'No cells found - check the .mat files under /content'

## Step 7 - Preprocessing
SOH smoothing, 60/40 split by cell, min-max from train cells only, window generator.

In [ ]:
# --------------------------- PREPROCESSING --------------------------------
cells = [c for c in cells if len(c['soh']) >= max(91, IN_STEP + OUT_STEP + 1)]
for c in cells:                                           # smoothing, as in repo
    c['soh'] = savgol_filter(c['soh'], 90, 2).astype(np.float32)
train_size = int(0.6 * len(cells))                        # first 60% of cells = train (repo)
tr_ids = list(range(train_size)); te_ids = list(range(train_size, len(cells)))
mn = np.min([c['mn'] for c in cells[:train_size]], axis=0)   # min-max from TRAIN cells only
mx = np.max([c['mx'] for c in cells[:train_size]], axis=0)
for c in cells:
    mask = np.arange(LEN)[None, None, :] < c['n'][:, :, None]
    c['X'] = np.where(mask, (c['X'] - mn[None, :, None]) / (mx - mn + 1e-9)[None, :, None], 0).astype(np.float32)
print(f'cells used: {len(cells)} (train {len(tr_ids)} / test {len(te_ids)}) | total cycles: '
      f'{sum(len(c["soh"]) for c in cells)} | lowest SOH reached: {min(c["soh"].min() for c in cells):.3f}')

def make_index(ids, stride=1):
    return [(c, i) for c in ids for i in range(0, len(cells[c]['soh']) - IN_STEP - OUT_STEP + 1, stride)]

class WinSeq(tf.keras.utils.Sequence):                    # windows built on the fly (no RAM blow-up)
    def __init__(self, idx, bs=64, mode='ae', shuffle=False):
        super().__init__()
        self.idx, self.bs, self.mode, self.shuffle = list(idx), bs, mode, shuffle
        self.on_epoch_end()
    def __len__(self): return int(np.ceil(len(self.idx) / self.bs))
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.idx)
    def __getitem__(self, b):
        sel = self.idx[b * self.bs:(b + 1) * self.bs]
        x = np.stack([cells[c]['X'][i:i + IN_STEP] for c, i in sel])
        return (x, x) if self.mode == 'ae' else x

## Step 8 - Train the auto-encoder (encoder part of E-LSTM)

In [ ]:
# ----------------------------- AUTO-ENCODER -------------------------------
inp = layers.Input((IN_STEP, 6, LEN))
h = layers.Flatten()(inp)
h = layers.Dense(256, activation='relu')(h)
h = layers.Dense(128, activation='relu')(h)
h = layers.Dense(100, activation='relu')(h)
code = layers.Dense(FSIZE * IN_STEP, activation='sigmoid')(h)          # 640-d bottleneck
d = layers.Dense(256, activation='relu')(code)
d = layers.Dense(128, activation='relu')(d)
d = layers.Dense(256, activation='relu')(d)
d = layers.Dense(6 * LEN * IN_STEP, activation='sigmoid')(d)
ae = models.Model(inp, layers.Reshape((IN_STEP, 6, LEN))(d))
encoder = models.Model(inp, code)
ae.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mae')
ae.fit(WinSeq(make_index(tr_ids, AE_STRIDE), 64, 'ae', True),
       validation_data=WinSeq(make_index(te_ids, 5), 64, 'ae'),
       epochs=AE_EPOCHS, verbose=2,
       callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)])

## Step 9 - Encode all windows (640-d code per window)

In [ ]:
# encode every window (train cells first, then test cells -> same order as repo's concat)
all_idx = make_index(tr_ids) + make_index(te_ids)
E = encoder.predict(WinSeq(all_idx, 256, 'x'), verbose=1).astype(np.float32).reshape(-1, IN_STEP, FSIZE)
Y = np.stack([cells[c]['soh'][i + IN_STEP:i + IN_STEP + OUT_STEP] for c, i in all_idx]).astype(np.float32)
grp = np.array([c for c, _ in all_idx])
print('encoded windows:', E.shape, 'targets:', Y.shape)

## Step 10 - Define the LSTM

In [ ]:
# --------------------------------- LSTM -----------------------------------
def make_lstm():
    m = models.Sequential([
        layers.Input((IN_STEP, FSIZE)),
        layers.LSTM(100, activation='relu', return_sequences=True, kernel_initializer='glorot_uniform'),
        layers.LSTM(100, activation='relu', kernel_initializer='glorot_uniform'),
        layers.Dense(OUT_STEP, activation='sigmoid', kernel_initializer='glorot_uniform')])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mae')
    return m

def mape(y, p): return np.mean(np.abs((y - p) / y)) * 100

## Step 11 - 5-fold cross-validation and comparison with the paper
Run once with `'paper'`, then change to `'cell'` and run this cell + Step 12 again. Report both.

In [ ]:
SPLIT_MODE = 'paper'   # 'paper' = authors' protocol | 'cell' = honest split by battery cell
# (honest run: change to 'cell' and re-run ONLY this cell and the plot cell below)

if SPLIT_MODE == 'paper':
    splits = list(KFold(5, shuffle=True, random_state=42).split(E))
    print('NOTE: paper protocol = random K-fold over overlapping windows, so neighbouring '
          'windows of the same cell land in both train and validation.')
else:
    splits = list(GroupKFold(5).split(E, groups=grp))

res, fold_models = [], []
for f, (tr, va) in enumerate(splits):
    m = make_lstm()
    m.fit(E[tr], Y[tr], epochs=LSTM_EPOCHS, verbose=0)                  # batch 32 = repo default
    p = m.predict(E[va], verbose=0)
    mae = np.mean(np.abs(Y[va] - p)); mse = np.mean((Y[va] - p) ** 2)
    res.append([mae * 100, mse * 1e3, np.sqrt(mse) * 100, mape(Y[va], p)])
    fold_models.append(m)
    print(f'Fold {f+1}/5: MAE={res[-1][0]:.2f}e-2  MSE={res[-1][1]:.3f}e-3  '
          f'RMSE={res[-1][2]:.2f}e-2  MAPE={res[-1][3]:.2f}%')

res = np.array(res); mine = res.mean(0)
ref = PAPER.get((IN_STEP, OUT_STEP))
print(f'\n=== E-LSTM, window {(IN_STEP, OUT_STEP)}, split={SPLIT_MODE} (mean +- std over 5 folds) ===')
print(f'{"metric":14s}{"mine":>16s}{"paper":>10s}')
for n_, a, s, b in zip(['MAE x1e-2', 'MSE x1e-3', 'RMSE x1e-2', 'MAPE %'], mine, res.std(0),
                       ref if ref else [np.nan] * 4):
    print(f'{n_:14s}{a:9.2f} +-{s:5.2f}{b:10.2f}')

## Step 12 - Fig. 8-style graph

In [ ]:
# --------------------- Fig.8-style plot (fold-4 model, like repo 'F3') ------
va3 = splits[3][1]
cand = sorted(set(grp[va3])) if SPLIT_MODE == 'cell' else te_ids
ci = min(cand, key=lambda k: cells[k]['soh'].min())                    # most degraded candidate cell
comb = models.Model(inp, fold_models[3](layers.Reshape((IN_STEP, FSIZE))(encoder(inp))))
c = cells[ci]; xs, pr, s0 = [], [], 0
while s0 + IN_STEP + OUT_STEP <= len(c['soh']):
    pr += list(comb.predict(c['X'][s0:s0 + IN_STEP][None], verbose=0)[0])
    xs += list(range(s0 + IN_STEP, s0 + IN_STEP + OUT_STEP)); s0 += OUT_STEP
plt.figure(figsize=(8, 5))
plt.plot(c['soh'], 'k', label='true values')
plt.plot(xs, pr, 'b', label='prediction values of Encoder_lstm model')
plt.axvline(IN_STEP, color='orange', ls='--'); plt.text(IN_STEP + 5, 0.9, 'Prediction start', color='orange')
plt.xlabel('cycle number'); plt.ylabel('SOH'); plt.legend()
plt.title(f'{IN_STEP}-step input window, {OUT_STEP}-step ahead prediction')
plt.savefig(f'fig8_repro_{SPLIT_MODE}.png', dpi=200, bbox_inches='tight'); plt.show()